# Automated Medical Image Analysis for Early Disease Detection
## Phase 3: Cloud GPU Training Pipeline for YOLOv8 Pneumonia Detection

This notebook is designed for execution in **Kaggle Notebooks (GPU T4 x2 or P100)** with **Google Colab (T4 GPU)** as secondary.

### Pipeline Overview:
1. **Hardware & Environment**: CUDA acceleration, multi-core setup, library installations.
2. **Patient-Level Dataset Splitting**: 80% Train, 10% Validation, 10% Test with **zero patient data leakage**.
3. **High-Speed DICOM to YOLO Conversion**: VOI LUT, MONOCHROME inversion, 640x640 JPEG rendering, empty labels for negative normal radiographs.
4. **Integrity Validation**: Automated check for image-label pairing, box boundaries, and disjoint patient partitions.
5. **Lightweight Model Training**: Pretrained YOLOv8 (`yolov8s.pt` / `yolov8n.pt`) with class `pneumonia_opacity`.
6. **Persistent Checkpointing & Resume**: Save every epoch, preserve `best.pt`, with resume capability.
7. **Evaluation Metrics**: Precision, Recall, mAP@0.50, mAP@0.50:0.95, and **separate image-level classification metrics** (Accuracy, Sensitivity, Specificity).
8. **Diagnostic Artifacts**: Training loss curves, confusion matrices, and validation visual overlays.

### 1. Environment Setup & Hardware Check

In [ ]:
!nvidia-smi
!pip install -q ultralytics pydicom opencv-python-headless scikit-learn matplotlib pandas numpy pyyaml tqdm

In [ ]:
import os
import sys
import json
import yaml
import shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import torch
import pydicom
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Accelerator : {torch.cuda.get_device_name(0)}")
    print(f"Device Count    : {torch.cuda.device_count()}")
else:
    print("[WARNING] CUDA is not available. Please toggle Accelerator -> GPU in notebook settings!")

### 2. Locate RSNA Competition Dataset

In [ ]:
KAGGLE_PATH = Path("/kaggle/input/rsna-pneumonia-detection-challenge")
COLAB_PATH = Path("./data/raw")

INPUT_DATA_DIR = KAGGLE_PATH if KAGGLE_PATH.exists() else COLAB_PATH
labels_csv_path = INPUT_DATA_DIR / "stage_2_train_labels.csv"
images_dicom_dir = INPUT_DATA_DIR / "stage_2_train_images"

assert labels_csv_path.exists(), f"Labels CSV not found at {labels_csv_path}! Attach RSNA competition data."
print(f"Found RSNA dataset at: {INPUT_DATA_DIR}")

df_labels = pd.read_csv(labels_csv_path)
print(f"Total annotation rows: {len(df_labels):,}")
print(f"Unique patients      : {df_labels['patientId'].nunique():,}")
display(df_labels.head())

### 3. Patient-Level Stratified Train / Val / Test Split (Zero Patient Leakage)

In [ ]:
def split_patients_stratified(df, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10, seed=42):
    patient_targets = df.groupby("patientId")["Target"].max()
    rng = np.random.RandomState(seed)
    split_map = {}
    
    for target_val in [0, 1]:
        pts = patient_targets[patient_targets == target_val].index.to_numpy()
        rng.shuffle(pts)
        n = len(pts)
        n_tr = int(round(n * train_ratio))
        n_va = int(round(n * val_ratio))
        
        for pid in pts[:n_tr]:
            split_map[pid] = "train"
        for pid in pts[n_tr:n_tr + n_va]:
            split_map[pid] = "val"
        for pid in pts[n_tr + n_va:]:
            split_map[pid] = "test"
            
    return split_map

patient_splits = split_patients_stratified(df_labels, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10)

# Verification of zero patient leakage
tr_pts = {p for p, s in patient_splits.items() if s == 'train'}
va_pts = {p for p, s in patient_splits.items() if s == 'val'}
te_pts = {p for p, s in patient_splits.items() if s == 'test'}

assert len(tr_pts.intersection(va_pts)) == 0, "LEAKAGE: Train & Val share patients!"
assert len(tr_pts.intersection(te_pts)) == 0, "LEAKAGE: Train & Test share patients!"
assert len(va_pts.intersection(te_pts)) == 0, "LEAKAGE: Val & Test share patients!"

print("[PASS] Zero patient data leakage confirmed!")
print(f"Train patients: {len(tr_pts):,} | Val patients: {len(va_pts):,} | Test patients: {len(te_pts):,}")

### 4. High-Speed DICOM to YOLO Image & Label Generation
- Decodes DICOM 16-bit radiographs with VOI LUT and `MONOCHROME1` inversion.
- Resizes to 640x640 JPEG for optimal GPU throughput.
- Generates `.txt` label files for class `0: pneumonia_opacity`.
- Normal cases (`Target=0`) generate **empty** `.txt` files for negative background learning.

In [ ]:
YOLO_DATA_DIR = Path("/kaggle/working/rsna_yolo") if Path("/kaggle/working").exists() else Path("./data/processed/rsna_yolo")

for s in ["train", "val", "test"]:
    (YOLO_DATA_DIR / "images" / s).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)

from pydicom.pixel_data_handlers.util import apply_voi_lut

def process_single_patient(args):
    patient_id, split, bboxes = args
    dcm_file = images_dicom_dir / f"{patient_id}.dcm"
    out_img = YOLO_DATA_DIR / "images" / split / f"{patient_id}.jpg"
    out_lbl = YOLO_DATA_DIR / "labels" / split / f"{patient_id}.txt"
    
    # 1. Convert DICOM -> 640x640 JPEG
    if not out_img.exists() and dcm_file.exists():
        dcm = pydicom.dcmread(str(dcm_file))
        try:
            arr = apply_voi_lut(dcm.pixel_array, dcm)
        except Exception:
            arr = dcm.pixel_array
            
        if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = np.amax(arr) - arr
            
        arr = arr.astype(float)
        mn, mx = np.min(arr), np.max(arr)
        norm = ((arr - mn) / (mx - mn) * 255.0).astype(np.uint8) if mx > mn else np.zeros(arr.shape, dtype=np.uint8)
        resized = cv2.resize(norm, (640, 640), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(out_img), resized, [cv2.IMWRITE_JPEG_QUALITY, 92])
        
    # 2. Convert bounding boxes to normalized YOLO format
    lbl_lines = []
    for (x, y, w, h) in bboxes:
        xc = (x + w / 2.0) / 1024.0
        yc = (y + h / 2.0) / 1024.0
        wn = w / 1024.0
        hn = h / 1024.0
        lbl_lines.append(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
        
    out_lbl.write_text("\n".join(lbl_lines) + ("\n" if lbl_lines else ""))
    return True

# Build task list
grouped = df_labels.groupby("patientId")
tasks = []
for pid, s in patient_splits.items():
    boxes = []
    if pid in grouped.groups:
        for _, r in grouped.get_group(pid).iterrows():
            if r["Target"] == 1 and pd.notna(r["x"]):
                boxes.append((float(r["x"]), float(r["y"]), float(r["width"]), float(r["height"])))
    tasks.append((pid, s, boxes))

print(f"Prepared {len(tasks):,} patients for conversion. Executing multi-core conversion...")
with ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
    list(tqdm(executor.map(process_single_patient, tasks), total=len(tasks), desc="Converting RSNA to YOLO"))

print("[SUCCESS] Conversion completed!")

### 5. Generate `rsna_yolo.yaml` and Run Dataset Integrity Validation

In [ ]:
yaml_config = {
    "path": str(YOLO_DATA_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": {0: "pneumonia_opacity"}
}

yaml_path = YOLO_DATA_DIR / "rsna_yolo.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_config, f, default_flow_style=False, sort_keys=False)

print(f"Dataset YAML created at {yaml_path}:")
print(open(yaml_path).read())

# Verification Check
for split in ["train", "val", "test"]:
    n_imgs = len(list((YOLO_DATA_DIR / "images" / split).glob("*.jpg")))
    n_lbls = len(list((YOLO_DATA_DIR / "labels" / split).glob("*.txt")))
    print(f"{split.capitalize():<5} -> Images: {n_imgs:,} | Labels: {n_lbls:,}")
    assert n_imgs == n_lbls, f"Mismatch in {split}: {n_imgs} images vs {n_lbls} labels!"

print("[PASS] 100% paired image-label integrity validated across all partitions!")

### 6. Train YOLOv8 Detector with Persistent Checkpointing
- Model: `yolov8s.pt` (Small, lightweight, highly accurate for single-class opacity detection).
- `save=True`, `save_period=1`: Checkpoint saved after every single epoch.
- Output project: `/kaggle/working/rsna_pneumonia` (persistent working directory).
- Best weights preserved automatically as `best.pt`.

In [ ]:
# Set experiment directory
EXPERIMENT_DIR = Path("/kaggle/working/rsna_pneumonia") if Path("/kaggle/working").exists() else Path("./models/runs/rsna_pneumonia")
RUN_NAME = "yolov8s_pneumonia_run"

# Check for resume capability
last_checkpoint = EXPERIMENT_DIR / RUN_NAME / "weights" / "last.pt"

if last_checkpoint.exists():
    print(f"[RESUME] Found existing checkpoint: {last_checkpoint}. Resuming training...")
    model = YOLO(str(last_checkpoint))
    train_results = model.train(resume=True)
else:
    print("[INITIALIZE] Starting fresh YOLOv8 training with transfer learning...")
    model = YOLO("yolov8s.pt")
    train_results = model.train(
        data=str(yaml_path),
        epochs=25,
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else "cpu",
        project=str(EXPERIMENT_DIR),
        name=RUN_NAME,
        save=True,
        save_period=1,
        patience=10,
        exist_ok=True,
        verbose=True
    )

### 7. Comprehensive Model Evaluation
Evaluates bounding box detection (`Precision`, `Recall`, `mAP@0.50`, `mAP@0.50:0.95`) on both Validation and Holdout Test splits.

In [ ]:
best_weights = EXPERIMENT_DIR / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(str(best_weights))

# Validation split evaluation
print("=== OBJECT DETECTION METRICS (VALIDATION SPLIT) ===")
val_metrics = best_model.val(data=str(yaml_path), split="val", imgsz=640)
print(f"Precision (P)    : {val_metrics.box.p[0]:.4f}")
print(f"Recall (R)       : {val_metrics.box.r[0]:.4f}")
print(f"mAP @ 0.50       : {val_metrics.box.map50:.4f}")
print(f"mAP @ 0.50:0.95  : {val_metrics.box.map:.4f}")

# Holdout Test split evaluation
print("\n=== OBJECT DETECTION METRICS (HOLDOUT TEST SPLIT) ===")
test_metrics = best_model.val(data=str(yaml_path), split="test", imgsz=640)
print(f"Precision (P)    : {test_metrics.box.p[0]:.4f}")
print(f"Recall (R)       : {test_metrics.box.r[0]:.4f}")
print(f"mAP @ 0.50       : {test_metrics.box.map50:.4f}")
print(f"mAP @ 0.50:0.95  : {test_metrics.box.map:.4f}")

### 8. Separate Image-Level Classification Evaluation
Calculates clinical patient-level diagnosis metrics (Sensitivity, Specificity, Accuracy, and Classification Confusion Matrix).

In [ ]:
# Evaluate image-level classification performance on Test split
test_images = list((YOLO_DATA_DIR / "images" / "test").glob("*.jpg"))
test_labels = (YOLO_DATA_DIR / "labels" / "test")

y_true = []
y_pred = []

print(f"Evaluating image-level classification on {len(test_images):,} test radiographs...")
for img_p in tqdm(test_images[:500], desc="Classification Evaluation"):
    lbl_p = test_labels / f"{img_p.stem}.txt"
    gt_has_pneumonia = 1 if lbl_p.exists() and len(lbl_p.read_text().strip()) > 0 else 0
    
    preds = best_model.predict(source=str(img_p), conf=0.25, verbose=False)
    pred_has_pneumonia = 1 if (preds and len(preds[0].boxes) > 0) else 0
    
    y_true.append(gt_has_pneumonia)
    y_pred.append(pred_has_pneumonia)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

tp = np.sum((y_true == 1) & (y_pred == 1))
fp = np.sum((y_true == 0) & (y_pred == 1))
tn = np.sum((y_true == 0) & (y_pred == 0))
fn = np.sum((y_true == 1) & (y_pred == 0))

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0

print("\n=== CLINICAL IMAGE-LEVEL CLASSIFICATION METRICS ===")
print(f"Accuracy          : {accuracy:.2%}")
print(f"Sensitivity/Recall: {sensitivity:.2%} (True Positive Rate)")
print(f"Specificity       : {specificity:.2%} (True Negative Rate)")
print(f"Confusion Matrix  : TP={tp}, FP={fp}, TN={tn}, FN={fn}")

### 9. Visual Inspection & Training Artifact Plots

In [ ]:
run_dir = EXPERIMENT_DIR / RUN_NAME

# Display training progression curve
results_plot = run_dir / "results.png"
if results_plot.exists():
    print("Training Loss & mAP Curves:")
    img_plot = cv2.imread(str(results_plot))
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

# Display detection confusion matrix
cm_plot = run_dir / "confusion_matrix.png"
if cm_plot.exists():
    print("Detection Confusion Matrix:")
    img_cm = cv2.imread(str(cm_plot))
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img_cm, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

### 10. Download `best.pt` for Apple M1 Deployment

Once training completes, the fine-tuned model checkpoint is saved at:
```text
/kaggle/working/rsna_pneumonia/yolov8s_pneumonia_run/weights/best.pt
```

1. Go to the **Output** tab in the Kaggle right sidebar.
2. Locate `best.pt` under `rsna_pneumonia/yolov8s_pneumonia_run/weights/`.
3. Click **Download** and place it in your local repository under:
   `models/weights/best.pt`.
4. Your Apple M1 Mac can now run local inference with Metal Performance Shaders (`mps`)!